In [0]:
# -------------------------------------------------------------------------
# BRONZE LAYER QUALITY TESTS (DLT EDITION)
# Purpose: Verify DLT Ingestion, Audit Columns, and Error Handling
# Rubric: Domain 3 (Data Quality, Error Handling) & Domain 4 (Audit)
# -------------------------------------------------------------------------

from pyspark.sql.functions import col, count

# 1. Setup
catalog = "ecommerce_analytics_dev"
schema = "bronze_layer"
table_name = f"{catalog}.{schema}.events_raw"

print(f"🧪 Starting Quality Tests for: {table_name}")

# 2. TEST 1: Table Accessibility
try:
    df = spark.read.table(table_name)
    print("✅ TEST 1 PASSED: Table is accessible.")
except Exception as e:
    print(f"❌ TEST 1 FAILED: Cannot read table. Run the DLT pipeline first! Error: {e}")

# 3. TEST 2: Record Count (Sanity Check)
row_count = df.count()
if row_count > 0:
    print(f"✅ TEST 2 PASSED: Table is populated.")
else:
    print("❌ TEST 2 FAILED: Table is empty.")

# 4. TEST 3: Schema Validation (Critical Columns)
# We check for standard columns AND the DLT-specific _rescued_data
expected_columns = {
    "event_time": "timestamp",
    "user_id": "integer",
    "price": "double",
    "source_file": "string",          # Audit Column Check
    "ingestion_timestamp": "timestamp", # Audit Column Check
    "_rescued_data": "string"         # DLT Error Handling Check
}

print("--- Checking Schema ---")
current_schema = {f.name: f.dataType.typeName() for f in df.schema}
all_cols_pass = True

for col_name, expected_type in expected_columns.items():
    if col_name not in current_schema:
        print(f"❌ FAILED: Column '{col_name}' is MISSING.")
        all_cols_pass = False
    elif current_schema[col_name] != expected_type:
        # Note: _rescued_data might be complex string/struct, simplified check here
        print(f"⚠️ WARNING: Column '{col_name}' has type '{current_schema[col_name]}', expected '{expected_type}'.")
    else:
        print(f"   Column '{col_name}' exists and is {expected_type}.")

if all_cols_pass:
    print("✅ TEST 3 PASSED: Critical schema columns are present.")

# 5. TEST 4: Audit Column Verification
# Rubric Domain 4: "Raw data with audit columns"
null_audit_count = df.filter(col("ingestion_timestamp").isNull() | col("source_file").isNull()).count()

if null_audit_count == 0:
    print("✅ TEST 4 PASSED: Audit columns (ingestion_timestamp, source_file) are fully populated.")
else:
    print(f"❌ TEST 4 FAILED: Found {null_audit_count} rows with missing audit info.")

# 6. TEST 5: Volume Verification (13GB Check)
print(f"\n📊 Volume Check: {row_count:,.0f} total rows.")

if row_count > 1000000:
    print("✅ TEST 5 PASSED: Volume looks healthy (Millions of records).")
elif row_count == 0:
    print("❌ TEST 5 FAILED: Zero records found.")
else:
    print("⚠️ WARNING: Row count is low (< 1M). Did the stream process the full 13GB file?")

# 7. TEST 6: DLT Error Handling Check (_rescued_data)
# Rubric Domain 3: "Error Handling and Dead Letter Queues"
if "_rescued_data" in df.columns:
    rescued_count = df.filter(col("_rescued_data").isNotNull()).count()
    print(f"✅ TEST 6 PASSED: DLT Error handling column (_rescued_data) exists.")
    if rescued_count > 0:
        print(f"   ℹ️ Info: {rescued_count} rows were rescued (malformed data found and saved).")
    else:
        print(f"   ℹ️ Info: 0 rows rescued (Input data was perfectly formatted).")
else:
    print("❌ TEST 6 FAILED: _rescued_data column missing. DLT error handling not active.")

# 8. Sample Data Display
print("\n--- Sample Data (First 5 Rows) ---")
display(df.limit(5))

In [0]:

# -------------------------------------------------------------------------
# DATA PROFILE: NULL VALUE ANALYSIS
# Purpose: Scan all 110M rows to find which columns actually have missing data.
# -------------------------------------------------------------------------

from pyspark.sql.functions import col, count, when

# 1. Define Table
table_name = "ecommerce_analytics_dev.bronze_layer.events_raw"
print(f"📊 Analyzing Nulls for: {table_name}")

# 2. Read Data
df = spark.read.table(table_name)
total_rows = df.count()
print(f"   Total Rows: {total_rows:,.0f}")

# 3. Dynamic Null Check (Scans ALL columns automatically)
# This creates a list of expressions: count(when(col is null, 1))
null_expressions = [
    count(when(col(c).isNull(), c)).alias(c) 
    for c in df.columns
]

# 4. Execute Aggregation (This triggers the computation)
# We collect the result to the driver to print nicely
print("   Running heavy computation... (This might take 1-2 mins)")
null_counts = df.select(null_expressions).collect()[0].asDict()

# 5. Display Results
print("\n--- 🔍 NULL COUNT REPORT ---")
print(f"{'Column Name':<25} | {'Null Count':<15} | {'Percentage'}")
print("-" * 60)

for col_name, null_val in null_counts.items():
    if null_val > 0:
        percent = (null_val / total_rows) * 100
        # Highlight in RED if it's a critical column, YELLOW if non-critical
        marker = "⚠️" if percent > 0 else "✅"
        print(f"{marker} {col_name:<22} | {null_val:<15,.0f} | {percent:.2f}%")
    else:
        print(f"✅ {col_name:<22} | 0               | 0.00%")

print("-" * 60)

In [0]:
# -------------------------------------------------------------------------
# SILVER LAYER QUALITY VALIDATION
# Purpose: Verify Smart Filling (Imputation), Deduplication, and Quality Rules
# -------------------------------------------------------------------------

from pyspark.sql.functions import col, count, when

# 1. Configuration - Point to your specific SILVER SCHEMA table
# NOTE: Ensure this matches the 'Target Schema' you set in your Silver Pipeline
silver_table_name = "ecommerce_analytics_dev.silver_layer.events_silver"
quarantine_table_name = "ecommerce_analytics_dev.silver_layer.events_quarantine"

print(f"🧪 Starting Silver Validation for: {silver_table_name}")

try:
    df = spark.read.table(silver_table_name)
    total_rows = df.count()
    print(f"✅ Table Accessed. Total Clean Records: {total_rows:,.0f}")
except Exception as e:
    print(f"❌ CRITICAL FAIL: Cannot read table. Check Schema Name! Error: {e}")
    dbutils.notebook.exit("Stop")

# -------------------------------------------------------------------------
# TEST 1: NULL HANDLING (The "Smart Fill" Check)
# -------------------------------------------------------------------------
print("\n--- 🔍 TEST 1: Smart Null Handling ---")
# We expect 0 NULLS in these columns because we used coalesce() to fill them.
# If we see NULLs here, the code failed.
null_counts = df.select(
    count(when(col("brand").isNull(), 1)).alias("brand_nulls"),
    count(when(col("category_code").isNull(), 1)).alias("category_nulls"),
    count(when(col("brand") == "Unknown", 1)).alias("brand_filled"),
    count(when(col("category_code") == "n/a", 1)).alias("category_filled")
).collect()[0]

if null_counts["brand_nulls"] == 0 and null_counts["category_nulls"] == 0:
    print("✅ PASS: No Nulls found in 'brand' or 'category_code'.")
    print(f"   ℹ️  Info: {null_counts['brand_filled']:,.0f} rows were filled with 'Unknown'.")
    print(f"   ℹ️  Info: {null_counts['category_filled']:,.0f} rows were filled with 'n/a'.")
else:
    print("❌ FAIL: Found NULLS in columns that should be filled!")
    print(null_counts)

# -------------------------------------------------------------------------
# TEST 2: CRITICAL DATA QUALITY (The "Expectations" Check)
# -------------------------------------------------------------------------
print("\n--- 🔍 TEST 2: Critical Rules ---")
# We dropped bad user_ids and event_times. These counts must be ZERO.
critical_errors = df.select(
    count(when(col("user_id").isNull(), 1)).alias("bad_users"),
    count(when(col("event_time").isNull(), 1)).alias("bad_times"),
    count(when(col("price") < 0, 1)).alias("negative_prices")
).collect()[0]

if (critical_errors["bad_users"] + critical_errors["bad_times"] + critical_errors["negative_prices"]) == 0:
    print("✅ PASS: Critical Quality Rules enforced (0 errors in clean data).")
else:
    print(f"❌ FAIL: Found bad data in Silver! {critical_errors}")

# -------------------------------------------------------------------------
# TEST 3: DEDUPLICATION CHECK
# -------------------------------------------------------------------------
print("\n--- 🔍 TEST 3: Deduplication Verify ---")
# This runs a quick check on a sample to ensure IDs are unique per time/event
# Running on full 110M might take too long, so we check logic on specific keys
duplicate_check = df.groupBy("event_time", "user_id", "product_id", "event_type").count().filter("count > 1")
dup_count = duplicate_check.count()

if dup_count == 0:
    print("✅ PASS: No duplicates found based on business key.")
else:
    print(f"❌ FAIL: Found {dup_count} duplicate records.")

# -------------------------------------------------------------------------
# TEST 4: QUARANTINE TABLE
# -------------------------------------------------------------------------
print("\n--- 🔍 TEST 4: Quarantine Inspection ---")
try:
    df_q = spark.read.table(quarantine_table_name)
    q_count = df_q.count()
    if q_count == 0:
        print("✅ PASS: Quarantine table exists (empty implies perfect input source).")
    else:
        print(f"✅ PASS: Quarantine trapped {q_count} bad rows.")
        display(df_q.limit(5))
except Exception as e:
    print(f"⚠️ WARNING: Could not read Quarantine table: {e}")